# Contexto de la industria: streaming frente a soporte físico

Figura animada para el inicio de la presentación: ingresos globales de la música grabada, 1999-2025, por formato.

**Datos:** copiados del notebook `2A_CT_EDA.ipynb` de Come Together (celda del gráfico IFPI), reconstruidos del reporte IFPI. No se vuelven a generar. Se excluye la dimensión de conciertos y, como en el original, la sincronización.

**Dos momentos señalados:** el año en que el streaming supera al soporte físico y el año en que el total supera el máximo de la etapa previa.

In [ ]:
# PAQUETES

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
import imageio.v2 as imageio
from PIL import Image

In [ ]:
# ============================================================
# CONFIGURACION: salidas y tipografías de la presentación
# ============================================================

DIRECTORIO_SALIDA = Path("../imagenes_estética")
ARCHIVO_BASE = "ifpi_streaming_vs_fisico"

FUENTE_TITULO = "C:/Windows/Fonts/BRLNSDB.TTF"
FUENTE_MONO = "../fuentes/DejaVuSansMono.ttf"
FUENTE_MONO_BOLD = "../fuentes/DejaVuSansMono-Bold.ttf"
for ruta in (FUENTE_TITULO, FUENTE_MONO, FUENTE_MONO_BOLD):
    font_manager.fontManager.addfont(ruta)
TITULO = font_manager.FontProperties(fname=FUENTE_TITULO)
MONO = font_manager.FontProperties(fname=FUENTE_MONO)
MONO_BOLD = font_manager.FontProperties(fname=FUENTE_MONO_BOLD)

# Animación: 1920 x 1080 a 30 cuadros por segundo
CUADROS_POR_ANIO = 11   # 20% más lento que la primera versión (9)
CUADROS_PAUSA_HITO = 75
CUADROS_FINAL = 120
FPS = 30

In [ ]:
# ============================================================
# DATOS: copiados de 2A_CT_EDA.ipynb (Come Together), sin cambios
# Valores en US$ billions. Sin conciertos ni sincronización.
# ============================================================

ingresos = pd.DataFrame({
    "anio": list(range(1999, 2026)),
    "physical": [
        22.3, 20.9, 21.6, 20.0, 18.5, 18.0, 16.8, 15.1, 13.0,
        10.9, 9.5, 8.2, 7.4, 6.7, 6.0, 5.3, 5.1, 4.8,
        4.6, 4.0, 3.8, 3.7, 4.3, 4.4, 5.0, 4.9, 5.3
    ],
    "streaming": [
        0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.1, 0.2,
        0.3, 0.4, 0.4, 0.6, 0.9, 1.3, 1.8, 2.7, 4.4,
        6.2, 8.8, 10.7, 12.7, 15.6, 17.4, 19.1, 20.5, 22.0
    ],
    "downloads": [
        0.0, 0.0, 0.0, 0.0, 0.0, 0.3, 0.9, 1.8, 2.5,
        3.1, 3.4, 3.6, 4.0, 4.2, 4.1, 3.8, 3.6, 3.0,
        2.5, 1.6, 1.4, 1.2, 1.1, 0.9, 0.9, 0.8, 0.8
    ],
    "performance_rights": [
        0.0, 0.0, 0.6, 0.7, 0.7, 0.9, 0.9, 1.0, 1.1,
        1.2, 1.2, 1.3, 1.4, 1.5, 1.7, 1.8, 1.8, 2.1,
        2.2, 2.5, 2.4, 2.2, 2.3, 2.5, 2.8, 2.9, 2.9
    ],
})

FORMATOS = ["physical", "downloads", "performance_rights", "streaming"]
ingresos["total"] = ingresos[FORMATOS].sum(axis=1).round(1)
ingresos

In [ ]:
# ============================================================
# HITOS
# - quiebre: primer año en que el streaming supera al físico
# - récord: primer año en que el total supera el máximo de la
#   etapa previa (1999-2001, antes de la caída)
# - piso: año de menor ingreso total
# ============================================================

anio_quiebre = int(ingresos.loc[ingresos["streaming"] > ingresos["physical"], "anio"].min())
maximo_previo = ingresos.loc[ingresos["anio"] <= 2001, "total"].max()
anio_record = int(ingresos.loc[(ingresos["anio"] > 2001) & (ingresos["total"] > maximo_previo), "anio"].min())
anio_piso = int(ingresos.loc[ingresos["total"].idxmin(), "anio"])

# Sanity check: coincide con el guion (2017 y 2021)
assert anio_quiebre == 2017 and anio_record == 2021, (anio_quiebre, anio_record)
print(f"Quiebre: {anio_quiebre} | récord: {anio_record} (supera {maximo_previo} B de 1999) | piso: {anio_piso}")
ingresos.loc[ingresos["anio"].isin([1999, anio_piso, anio_quiebre - 1, anio_quiebre, anio_record, 2025])]

In [ ]:
# ============================================================
# FIGURA: un cuadro para una cantidad de años visibles y el
# crecimiento de la última barra (fracción 0 a 1)
# Escala de grises: streaming en negro (el protagonista), físico
# en gris medio, descargas y ejecución pública en grises claros.
# ============================================================

COLORES = {"physical": "#8C8C8C", "downloads": "#CFCFCF", "performance_rights": "#E6E6E6", "streaming": "#111111"}
ETIQUETAS = {"physical": "Soporte físico", "downloads": "Descargas", "performance_rights": "Ejecución pública", "streaming": "Streaming"}

def dibujar(anios_visibles, fraccion, hitos_visibles):
    fig = plt.figure(figsize=(19.2, 10.8), dpi=100, facecolor="white")
    ax = fig.add_axes([0.06, 0.10, 0.90, 0.68])
    datos = ingresos[ingresos["anio"] <= anios_visibles]
    for i, fila in datos.iterrows():
        escala = fraccion if fila["anio"] == anios_visibles else 1.0
        base = 0.0
        for f in FORMATOS:
            alto = fila[f] * escala
            ax.bar(fila["anio"], alto, bottom=base, width=0.78, color=COLORES[f], linewidth=0)
            base += alto
        if fila["anio"] == anios_visibles:
            ax.text(fila["anio"], base + 0.5, f"{fila['total'] * escala:.1f}", ha="center", va="bottom",
                    fontproperties=MONO_BOLD, fontsize=15, color="#111111")
    # línea del máximo previo
    ax.axhline(maximo_previo, color="#BDBDBD", linewidth=1.2, linestyle=(0, (4, 4)), zorder=0)
    ax.text(1998.6, maximo_previo + 0.35, f"máximo de la etapa previa: {maximo_previo:.1f}", fontproperties=MONO, fontsize=13, color="#6B6B6B")
    if "quiebre" in hitos_visibles:
        fila = ingresos[ingresos["anio"] == anio_quiebre].iloc[0]
        ax.annotate(f"{anio_quiebre}\nel streaming supera\nal soporte físico", xy=(anio_quiebre, fila["total"] + 0.3),
                    xytext=(anio_quiebre - 5.2, 24.5), fontproperties=TITULO, fontsize=26, color="#111111", ha="center",
                    arrowprops=dict(arrowstyle="-|>", color="#111111", lw=2, shrinkA=6, shrinkB=4))
    if "record" in hitos_visibles:
        fila = ingresos[ingresos["anio"] == anio_record].iloc[0]
        ax.annotate(f"{anio_record}\nsupera el máximo\nde la etapa previa", xy=(anio_record - 0.42, fila["total"] - 0.2),
                    xytext=(anio_record - 3.6, 31.0), fontproperties=TITULO, fontsize=26, color="#111111", ha="center",
                    arrowprops=dict(arrowstyle="-|>", color="#111111", lw=2, shrinkA=6, shrinkB=4))
    ax.set_xlim(1998.3, 2025.7)
    ax.set_ylim(0, 36)
    ax.set_xticks(range(2000, 2026, 5))
    for t in ax.get_xticklabels():
        t.set_fontproperties(MONO_BOLD); t.set_fontsize(18)
    ax.set_yticks(range(0, 36, 10))
    for t in ax.get_yticklabels():
        t.set_fontproperties(MONO); t.set_fontsize(15); t.set_color("#6B6B6B")
    ax.grid(axis="y", color="#EDEDED", linewidth=1); ax.set_axisbelow(True)
    for lado in ("top", "right", "left"):
        ax.spines[lado].set_visible(False)
    ax.spines["bottom"].set_color("#BDBDBD")
    ax.tick_params(length=0)
    fig.text(0.06, 0.93, "Ingresos globales de la música grabada", fontproperties=TITULO, fontsize=44, color="#111111")
    fig.text(0.06, 0.875, "Miles de millones de dólares por año, por formato (sin conciertos)", fontproperties=MONO, fontsize=19, color="#4A4A4A")
    # leyenda en una fila, con separación fija entre elementos
    for f, x in zip(["streaming", "physical", "downloads", "performance_rights"], [0.06, 0.175, 0.32, 0.445]):
        fig.patches.append(plt.Rectangle((x, 0.815), 0.016, 0.028, facecolor=COLORES[f], transform=fig.transFigure,
                                         linewidth=0.8, edgecolor="#9A9A9A" if f == "performance_rights" else COLORES[f]))
        fig.text(x + 0.022, 0.818, ETIQUETAS[f], fontproperties=MONO_BOLD if f == "streaming" else MONO, fontsize=16, color="#111111")
    fig.text(0.06, 0.03, "Fuente: IFPI, Global Music Report 2026 (datos reconstruidos en Come Together).",
             fontproperties=MONO, fontsize=13, color="#6B6B6B")
    fig.canvas.draw()
    imagen = np.asarray(fig.canvas.buffer_rgba())[:, :, :3].copy()
    plt.close(fig)
    return imagen

# Sanity check visual: cuadro final completo
cuadro_final = dibujar(2025, 1.0, {"quiebre", "record"})
Image.fromarray(cuadro_final).save(DIRECTORIO_SALIDA / f"{ARCHIVO_BASE}.png")
Image.fromarray(cuadro_final).resize((960, 540))

In [ ]:
# ============================================================
# ANIMACIÓN: las barras crecen año por año; pausa en cada hito
# ============================================================

cuadros = []
hitos = set()
for anio in range(1999, 2026):
    for k in range(1, CUADROS_POR_ANIO + 1):
        cuadros.append(dibujar(anio, k / CUADROS_POR_ANIO, hitos))
    if anio == anio_quiebre:
        hitos.add("quiebre"); cuadros += [dibujar(anio, 1.0, hitos)] * CUADROS_PAUSA_HITO
    if anio == anio_record:
        hitos.add("record"); cuadros += [dibujar(anio, 1.0, hitos)] * CUADROS_PAUSA_HITO
cuadros += [cuadros[-1]] * CUADROS_FINAL

imageio.mimsave(DIRECTORIO_SALIDA / f"{ARCHIVO_BASE}.mp4", cuadros, fps=FPS, macro_block_size=8)
# GIF liviano para vista previa: mitad de tamaño, un cuadro de cada tres
cuadros_gif = [np.asarray(Image.fromarray(c).resize((960, 540))) for c in cuadros[::3]]
imageio.mimsave(DIRECTORIO_SALIDA / f"{ARCHIVO_BASE}.gif", cuadros_gif, duration=3 / FPS, loop=0)

print(f"{len(cuadros)} cuadros, {len(cuadros) / FPS:.1f} segundos")